<a href="https://colab.research.google.com/github/rahulpanigrahy650-droid/deep_learning_1/blob/main/RNN_sentiment_analysisipynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd

In [ ]:
import pandas as pd

df = pd.read_csv(
    "/content/sample_data/IMDB Dataset.csv",
    engine="python",
    on_bad_lines="skip"
)

df.head()


,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [ ]:
df.shape

(769, 2)

In [ ]:
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [ ]:
df.isnull().sum()

,0
review,0
sentiment,0


In [ ]:
df.drop_duplicates(inplace=True)

In [ ]:
df.shape

(769, 2)

preprocessing

converting to lowercase

In [ ]:
df["review"] = df["review"].str.lower()

Removing urls using regular expression (Regex)


In [ ]:
import re

# sample_text = "abc is the word ,abc" # abc => xyz

# new_text = re.sub("abc" , "xyz" , sample_text)

In [ ]:
def remove_url(text):
  text = re.sub(r"https\S+","",text) # (pattern,repl,string) e.g - https://www.google.com
  return text

df["review"] = df["review"].apply(remove_url)

Removing punctuations

In [ ]:
def remove_punctuations(text):
  text = re.sub(r"[^A-Za-z0-9\s]","",text) # A-Z a-z 0-9 \s
  return text

df["review"] = df["review"].apply(remove_punctuations)

Removing HTML

In [ ]:
def remove_html(text):
    text = re.sub(r"<.*?>","",text)
    return text

df["review"] = df["review"].apply(remove_url)

Removing the stopwords


In [ ]:
import nltk

nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords")

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [ ]:
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords


In [ ]:
#sample_text = "I Like coding in python"
#tokens = word_tokenize(sample_text)

In [ ]:
#tokens

In [ ]:
def remove_stopwords(text):
  tokens = word_tokenize(text)
  stop_words = stopwords.words("english")

  for word in tokens:
    if word in stop_words:
      text = text.replace(word,"")
  return text

In [ ]:
df.head()

,review,sentiment
0,one of the other reviewers has mentioned that ...,positive
1,a wonderful little production br br the filmin...,positive
2,i thought this was a wonderful way to spend ti...,positive
3,basically theres a family where a little boy j...,negative
4,petter matteis love in the time of money is a ...,positive


stemming

In [ ]:
# running -> run
# played -> play
# porterStemming
from  nltk.stem import PorterStemmer

In [ ]:
def stemming(text):
  ps = PorterStemmer()
  stemmed_words = []


  tokens = word_tokenize(text)
  for token in tokens:
    stemmed_token = ps.stem(token)
    stemmed_words.append(stemmed_token)

  return " ".join(stemmed_words)

df["review"] = df["review"].apply(stemming)

In [ ]:
df.head()

,review,sentiment
0,one of the other review ha mention that after ...,positive
1,a wonder littl product br br the film techniqu...,positive
2,i thought thi wa a wonder way to spend time on...,positive
3,basic there a famili where a littl boy jake th...,negative
4,petter mattei love in the time of money is a v...,positive


Encoding

In [ ]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

df["sentiment"] = le.fit_transform(df["sentiment"])


In [ ]:
y = df["sentiment"]

In [ ]:
y

,sentiment
0,1
1,1
2,1
3,0
4,1
...,...
764,1
765,0
766,0
767,0


Vectorization

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tf = TfidfVectorizer()

x = tf.fit_transform(df["review"])

In [ ]:
x

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 104332 stored elements and shape (769, 13479)>

In [ ]:
from sklearn.model_selection import train_test_split

x_train,x_test,y_train,y_test = train_test_split(
    x,y,test_size =0.2,random_state =42
)

In [ ]:
x_train.shape

(615, 13479)

In [ ]:
x_test.shape

(154, 13479)

In [ ]:
import torch
from torch.utils.data import TensorDataset,DataLoader

In [ ]:
x_train = x_train.toarray()
x_test = x_test.toarray()

In [ ]:
train_set = TensorDataset(
    torch.from_numpy(x_train).float(),
    torch.from_numpy(y_train.values).float()
)

test_set = TensorDataset(
    torch.from_numpy(x_test).float(),
    torch.from_numpy(y_test.values).float()
)

In [ ]:
train_loader = DataLoader(train_set,shuffle = True,batch_size= 64)
test_loader =  DataLoader(test_set,shuffle = True,batch_size = 64)

Build our Rnn

In [ ]:
import torch.nn as nn
import torch.optim as optim

In [ ]:
class RNN(nn.Module):
  def __init__(self,input_size,hidden_size =128,num_layers=1):
    super().__init__()

    self.hidden_size = hidden_size
    self.num_layers = num_layers

    # RNN layer
    self.rnn = nn.RNN(input_size,hidden_size,num_layers,batch_first=True)

    # fully connected layer
    self.fc = nn.Linear(hidden_size,1)
  def forward(self,x):
    # optional => shape(num of layers,batch_size ,hidden_state)
    h0 = torch.zeros(self.num_layers,x.size(0),self.hidden_size)

    out,_ = self.rnn(x,h0)

    # 1st value = hidden state of all timesteps  => (batch,seq_len,hidden size)
    # 2nd value = final hidden state of last timestep

    out = self.fc(out[:,-1,:])
    return out

In [ ]:
input_size = x_train.shape[1]

model = RNN(input_size)

criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters())

Training the RNN

In [ ]:
epochs = 10

for epoch in range(epochs):
    model.train()

    for xb,yb in train_loader:
        optimizer.zero_grad()

        xb = xb.unsqueeze(1) # add singelton direction

        output = model(xb) # (batch_size,1)

        output = torch.sigmoid(output.squeeze()) # (batch_size,1)

        loss = criterion(output,yb) # comput loss

        loss.backward() # backprop
        optimizer.step() # weight update

print(f"epoch = {epoch+1}/{epochs} and loss= {loss.item}")


epoch = 10/10 and loss= <built-in method item of Tensor object at 0x7e282816d7c0>


In [ ]:
# evaluate

model.eval()

with torch.no_grad():
  correct_vals = 0
  tot_vals = 0

  for xb,yb in test_loader:
    xb = xb.unsqueeze(1)

    outputs = model(xb)
    predicted = (torch.sigmoid(outputs.squeeze()) > 0.5).float()

    tot_vals += yb.size(0)
    correct_vals += (predicted == yb).sum().item()

print(f"accuracy = {correct_vals/tot_vals*100}")


accuracy = 75.97402597402598
